# top10-missingness-simulation
7.4.25

One of the reviewers asked that we simulate some missingness into the top 10 DA proteins 
as identified by Savage et al and see if we can "recover" them as DA after we impute with Lupine.
I don't think this is a great experiment, but we're not doing a whole lot else this reviewer requested, so it's probably best to just do it. 

In [23]:
import pandas as pd
import numpy as np 
from Bio import SeqIO
from tqdm import tqdm

#### Configs 

In [24]:
# The unimputed joint quants matrix
joint_fname="/net/noble/vol2/home/lincolnh/data/quant-data/UMich-normalized/joint-quants-normalized-shifted.csv"
min_pres=18

# The Lupine recon matrix
lupine_recon_ensembled_path="/net/noble/vol2/home/lincolnh/code/2023_harris_deep_impute/results/2024-05-30_DE_sandbox/results/lupine-ensemble-imputed.csv"

# The Ensemble (GENCODEv44) fasta
ensembl_path="/net/noble/vol2/home/lincolnh/code/2023_harris_deep_impute/results/2023-11-13_UMich_dataset/fastas/"
ensembl_df="gencode.v44.pc_translations.fa"

# The HGNC database file
hgnc_database_path="/net/noble/vol2/home/lincolnh/data/quant-data/HGNC_database.txt"

# The metadata dictionary, previously created
meta_path="/net/noble/vol2/home/lincolnh/code/2023_harris_deep_impute/results/2024-05-10_metadata_mapping/meta-dict.csv"

cohort_ids=["CCRCC", "COAD", "HGSC", "HNSCC", "LSCC", "LUAD", "PDAC", "UCEC"]

# The fraction of missingness to simulate into the top-10 DA proteins
withhold_frac=0.4

rng = np.random.default_rng(seed=18)

#### Functions

In [25]:
def simulate_missingness_top10(joint_mat, mapper, cohort, mv_frac):
    """
    For a given CPTAC cohort, simulate some fraction of missingness into 
    only the top-10 DA proteins for that cohort, as per Savage et al. 
    
    Parameters
    ----------
    joint_mat : `pandas.DataFrame`, 
        The unimputed joint quantifications matrix. 
    mapper : dict, 
        The dictionary mapping CPTAC cohorts to the top-10 DA proteins 
        for each cohort. 
    cohort : str, 
        The CPTAC cohort. 
    mv_frac : float, 
        The fraction of present values to withhold 

    Returns
    ----------
    joint_mat : `pandas.DataFrame`, 
        Copy of the unimputed joint quants matrix. 
    """
    for i in range(0, joint_mat.shape[0]):
        curr_idx = joint_mat.index[i]
        curr_quants = np.array(joint_mat.iloc[i])
        
        if curr_idx in mapper[cohort]:
            n_mv = np.count_nonzero(np.isnan(curr_quants))
            n_pv = len(curr_quants) - n_mv
            n_withhold = int(n_pv * mv_frac)
            withhold_idx = rng.choice(list(range(0,len(curr_quants))), size=n_withhold)
            curr_quants[withhold_idx] = np.nan
    
            joint_mat.iloc[i] = curr_quants

    return joint_mat

#### Get the metadata, for a single cohort 
And get the tumor and nontumor sample IDs. 

In [26]:
# Might need `index_col=0` here
meta_dict = pd.read_csv(meta_path)

#### Pre-process the unimputed joint quants matrix

In [27]:
# Read in the joint quants matrix
joint_mat = pd.read_csv(joint_fname, index_col=0)

# Remove some of these extraneous runs
keywords = ["RefInt", "QC", "pool", "Tumor", "Pooled", 
            "Pool", "Reference", "NCI", "NX", "Ref"]
to_drop = []

for sample_id in list(joint_mat.columns):
    exclude=False
    for kw in keywords:
        if kw in sample_id:
            exclude=True
            break
    to_drop.append(exclude)

keep_cols = np.array(joint_mat.columns)[~np.array(to_drop)]
joint_mat = joint_mat[keep_cols]

joint = np.array(joint_mat)

# Remove proteins with too many missing values
num_present = np.sum(~np.isnan(joint), axis=1)
discard = num_present < min_pres
joint = np.delete(joint, discard, axis=0)
keep_prots = np.array(joint_mat.index)[~discard]

print(f"joint quants mat shape, post-filter: {joint.shape}")

joint_start = pd.DataFrame(joint, columns=keep_cols, index=keep_prots)

joint quants mat shape, post-filter: (18162, 1755)


#### Read in the Lupine-imputed joint quants matrix 

In [28]:
lupine_recon = pd.read_csv(lupine_recon_ensembled_path, index_col=0)
print(lupine_recon.shape)

(18162, 1755)


#### Create a dictionary mapping ENSPs to ENSGs

In [29]:
# Read in the HGNC database file
hgnc_db = pd.read_csv(hgnc_database_path, sep="\t")

# Read in the ENSEMBL fasta
ensembl_fasta = ensembl_path + ensembl_df
fasta_seqs = SeqIO.parse(open(ensembl_fasta), "fasta")

# Init both dictionaries
gene_x_prot = {}
prot_x_gene = {}

# Fill in the dictionary 
for fasta in fasta_seqs:
    name, descript, sequence = \
        fasta.id, fasta.description, str(fasta.seq)
    # Get the ENSP and ENSG IDs
    ensp_id = name.split("|")[0]
    ensg_id = name.split("|")[2]
    # Strip the ".x" characters. Hope this is ok.
    ensp_id = ensp_id.split(".")[0]
    ensg_id = ensg_id.split(".")[0]
    
    # Update the first dictionary
    prot_x_gene[ensp_id] = ensg_id
    
    # Update the second
    if ensg_id in gene_x_prot:
        gene_x_prot[ensg_id].append(ensp_id)
    else:
        gene_x_prot[ensg_id] = [ensp_id]

/tmp/ipykernel_561653/1917651185.py:2: DtypeWarning: Columns (31,38) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc_db = pd.read_csv(hgnc_database_path, sep="\t")


#### Create a dataframe mapping all of the names to one another 
ENSP > ENSG > HGNC

In [30]:
names_df = pd.DataFrame(columns=["ENSP", "ENSG", "HGNC"])
names_df["ENSP"] = lupine_recon.index

ensg_ids = []
hgnc_ids = []

for idx in range(0, names_df.shape[0]):
    curr = names_df.iloc[idx]
    curr_ensp = curr["ENSP"]
    
    try: 
        curr_ensg = prot_x_gene[curr_ensp]
    except KeyError:
        curr_ensg = None
        
    ensg_ids.append(curr_ensg)

    # Get the HGNC gene ID as well
    hgnc_id = None
    if curr_ensg is not None:
        try: 
            hgnc_row = hgnc_db[hgnc_db["ensembl_gene_id"] == curr_ensg]
            hgnc_id = hgnc_row["symbol"].item()
        except ValueError: 
            hgnc_id = None
            
    hgnc_ids.append(hgnc_id)

names_df["ENSG"] = ensg_ids
names_df["HGNC"] = hgnc_ids

#### Change the indices of the quants matrices to HGNC?
Is this a good idea? What about duplicates? And write this version of the unimputed joint quants matrix. 

In [31]:
joint_start.index = names_df["HGNC"]
joint_start.to_csv("../results/panCan-unimputed-starting-point.csv")

#### Create a dictionary to hold the top 10 DA proteins from Savage et al.

In [32]:
ccrcc_top10 = ["TBXAS1", "PKN1", "PYGL", "P4HB", "ITGA5", "PPAT", "CAD", "TRIO", "LDHA", "ACLY"]
luad_top10 = ["P4HB", "LDHA", "CDK12", "RANBP2", "SF3B1", "WDR5", "TOP1", "IARS1", "LARS1", "EEF2"]
coad_top10 = ["HSP90AB1", "CAD", "SLC3A2", "RANBP2", "TOP1", "SF3B1", "IARS1", "MARS1", "TOP2A"]
hgsc_top10 = ["ERP44", "APEX1", "HSP90AB1", "HSP90AA1", "PARP1", "TOP1", "HSP90B1", "MARS1", "IARS1", "SF3B1"]
hnscc_top10 = ["PTK7", "SLC38A1", "PTPN12", "FKBP9", "PAK2", "ITPR3", "PTPN1", "NMT1", "ATP6V1C1", "TOP1"]
pdac_top10 = ["ITGB4", "QSOX1", "ITPR3", "MET", "AEBP1", "NUCB1", "ITGAV", "LDHA", "GRK2", "LGALS3BP"]
lscc_top10 = ["GART", "CAD", "PARP1", "PPAT", "CDK12", "TOP1", "CDK9", "SF3B1", "EEF2", "RRM2"]
ucec_top10 = ["TACSTD2", "PIK3CB", "IL4I1", "KDM3A", "PPIF", "PTPRF", "PAK1", "IDE", "CSNK1A1", "MARS1"]

savage_mapper = {}

savage_mapper["CCRCC"] = ccrcc_top10
savage_mapper["LUAD"] = luad_top10
savage_mapper["COAD"] = coad_top10
savage_mapper["HGSC"] = hgsc_top10
savage_mapper["HNSCC"] = hnscc_top10
savage_mapper["PDAC"] = pdac_top10
savage_mapper["LSCC"] = lscc_top10
savage_mapper["UCEC"] = ucec_top10

#### What is the missingness fraction among the top-10 DA proteins for each cohort?
Prior to any imputation? 

In [33]:
for cohort in cohort_ids:
    meta_dict_curr = meta_dict[meta_dict["cohort"] == cohort]
    meta_dict_curr = meta_dict_curr.reset_index(drop=True)
    
    tumor_meta = meta_dict_curr[(meta_dict_curr["sample_type"] == "Primary Tumor") | (meta_dict_curr["sample_type"] == "Tumor")]
    nontumor_meta = meta_dict_curr[(meta_dict_curr["sample_type"] != "Primary Tumor") & (meta_dict_curr["sample_type"] != "Tumor")]
    
    tumor_IDs = list(tumor_meta["aliquot_ID"])
    nontumor_IDs = list(nontumor_meta["aliquot_ID"])
    curr_ids = tumor_IDs + nontumor_IDs
    
    unimputed_mat_cohort = joint_start[curr_ids]
    unimputed_mat_cohort = unimputed_mat_cohort.loc[savage_mapper[cohort]]
    
    nan_count = np.count_nonzero(np.isnan(unimputed_mat_cohort))
    nan_frac = nan_count / unimputed_mat_cohort.size
    print(f"{cohort} mv frac: {np.around(nan_frac, 3)}")

CCRCC mv frac: 0.181
COAD mv frac: 0.308
HGSC mv frac: 0.167
HNSCC mv frac: 0.0
LSCC mv frac: 0.091
LUAD mv frac: 0.182
PDAC mv frac: 0.289
UCEC mv frac: 0.213


#### Introduce additional missingness into the top-10 DA proteins
For all 10 cohorts at once? Or maybe just one at a time. I think the latter makes more sense. We can write 10 different matrices and impute them separately. 

In [22]:
for _cohort in tqdm(cohort_ids): 
    joint_mat_cohort = simulate_missingness_top10(joint_start, savage_mapper, _cohort, withhold_frac)
    joint_mat_cohort.index = joint_start.index
    #print(joint_mat_cohort.shape)
    
    #mv_count = np.count_nonzero(np.isnan(joint_mat_cohort))
    #mv_frac = mv_count / joint_mat_cohort.size
    #print(np.around(mv_frac,2))

    # Write the full joint quants matrix
    joint_mat_cohort.to_csv("../results/joint-quants-unimputed-"+_cohort+"-top10-sim.csv")

    # Write the single cohort quants matrix
    curr_meta = meta_dict[meta_dict["cohort"] == _cohort]
    curr_meta = curr_meta.reset_index(drop=True)
    curr_ids = list(curr_meta["aliquot_ID"])
    cohort_mat = joint_mat_cohort[curr_ids]
    #print(cohort_mat.shape)
    #cohort_mat.to_csv("../data/"+_cohort+"-quants-top10-sim.csv")

 62%|████████████████████████▍              | 5/8 [04:04<02:26, 48.97s/it]


KeyboardInterrupt: 